# T-REX 한식 인식 모델 학습 (YOLOv8n → TFLite)

AI Hub **74번 "음식 이미지 및 영양정보 텍스트"**(박스 어노테이션 포함, 400여 종)로 YOLOv8n을 파인튜닝하고, 앱(`app/src/main/assets/models/`)에 넣을 수 있는 형태로 변환하는 노트북이다. 모델은 휴대폰 온디바이스(LiteRT INT8) 기준이라 YOLOv8n 을 유지한다.

### 실행 전 준비 (1회)
1. [AI Hub](https://aihub.or.kr) 74번 활용신청 → 승인 → [aihubshell 안내 페이지](https://www.aihub.or.kr/devsport/apishell/list.do)의 **API key 발급** 버튼(키는 이메일로 온다). 브라우저 Innorix 는 쓰지 않는다.
2. Colab 런타임 유형: **G4 GPU + 고용량 RAM**(안 잡히면 A100 → L4). TPU 는 안 된다.
3. 아래 셀을 위에서부터 순서대로 실행. 4번 셀은 2b 출력(JSON 구조)을 본 뒤 완성한다.

### 데이터 구조 (2026-09-16 API 목록 기준)와 디스크 전략

| 항목 | 구성 | 크기 | 쓰임 |
|---|---|---|---|
| 라벨 TRAIN / VAL | `음식분류_라벨링_TRAIN_1223_add.zip` / `…_VAL_1223_add.zip` | 446MB / 56MB | 클래스·박스 JSON. **먼저 받는다** |
| **원천 Validation** | 대분류별 7묶음: `01`, `02_03_05`, `04`, `06_07_08`, `09_10`, `11`, `12_13_14_15_16` | 15~31GB씩, 합 177GB | **첫 학습 데이터.** 음식 분류를 골라 받을 수 있는 유일한 묶음 — 3~4묶음이면 클래스당 200장 안팎 |
| 원천 Training | `음식분류_이미지_TRAIN_001~050.zip` | 22~32GB × 50 ≈ 1.4TB | 번호가 클래스와 무관해 골라 받을 수 없다. 정확도가 더 필요할 때 몇 덩어리만 추가 |
| 영양 DB | `44.음식분류 AI 데이터 영양DB.xlsx` | 66KB | 앱 `foodDatabase` 근사값을 실제 값으로 교체 |
| 양추정 | `양추정_*` | 43GB+ | 인분/양 추정용 — **범위 밖** |

Validation 원천을 학습에 쓰고 train/val 은 우리가 다시 나눈다(공식 split 을 지키는 것보다 클래스를 골라 받는 이득이 크다). 묶음 하나씩 받아 목표 클래스만 640px 로 줄여 저장하고 원본을 지우므로 디스크 피크는 약 60GB, 완성 데이터셋은 3~5GB(Drive 보존).

### 산출물 (마지막 셀에서 zip으로 다운로드)
- `yolov8n_food.tflite` — INT8 양자화, **입출력은 float32 유지**. 앱 FoodDetector 는 NHWC/NCHW 입력을 모두 처리한다.
- `food_labels.txt` — 모델 클래스 인덱스 순서와 동일한 라벨 목록

두 파일을 `TREX_UI/app/src/main/assets/models/`에 덮어쓰고 빌드하면 앱이 자동으로 실추론으로 전환된다.

In [ ]:
# 1. 환경 설치 + 배정된 런타임 확인
# Colab 은 세션마다 GPU 가 다르게 배정된다(G4/H100/A100/L4/T4). 무엇을 받았는지 여기서 확인한다 — 아래 설정은 어느 GPU 든 그대로 동작한다.
!pip install -q -U ultralytics
import torch, ultralytics
ultralytics.checks()
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, VRAM {p.total_memory / 1e9:.0f} GB')
else:
    print('⚠️ GPU 가 배정되지 않았다 — 런타임 → 런타임 유형 변경에서 GPU 를 고른다')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -g | head -2
!df -h /content | tail -1

In [ ]:
# 2. AI Hub 다운로더(aihubshell) 설치 + 파일 목록 조회
AIHUB_API_KEY = ''   # aihubshell 안내 페이지의 "API key 발급" 버튼으로 받은 키 (이메일로 옴)
DATASET_KEY = 74     # 음식 이미지 및 영양정보 텍스트

assert AIHUB_API_KEY, 'AIHUB_API_KEY 를 채운다'
!curl -s -o aihubshell https://api.aihub.or.kr/api/aihubshell.do && chmod +x aihubshell && mv aihubshell /usr/local/bin/
!aihubshell -mode l -datasetkey {DATASET_KEY} -aihubapikey "{AIHUB_API_KEY}"

In [ ]:
# 3. 라벨 zip(TRAIN 446MB + VAL 56MB) + 영양 DB(xlsx) 다운로드 → 구조 확인
# ⛔ 처음 실행할 때는 여기까지만 돌리고, 아래에 출력되는 "== 받은 항목 ==" 부터 끝까지를 전달한다. 4번 이후는 아직 실행하지 않는다.
# filekey 는 2번 목록 기준(2026-09-16): 44897 = 음식분류_라벨링_TRAIN, 44878 = 음식분류_라벨링_VAL, 44887 = 영양DB.xlsx
LABEL_FILEKEYS = '44897,44878,44887'

!mkdir -p /content/aihub_raw && cd /content/aihub_raw && aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {LABEL_FILEKEYS} -aihubapikey "{AIHUB_API_KEY}"

import json
from pathlib import Path
raw = Path('/content/aihub_raw')
print('== 받은 항목 ==')
for p in sorted(raw.rglob('*')):
    if p.is_file() and p.suffix.lower() in {'.zip', '.xlsx'}:
        print(f'{p.relative_to(raw)}  {p.stat().st_size / 1e6:.0f} MB')
jsons = sorted(raw.rglob('*.json'))
print(f'== 풀린 JSON 파일 {len(jsons)}개 ==')
if jsons:
    print('예시:', [str(j.relative_to(raw)) for j in jsons[:5]])
    data = json.loads(jsons[0].read_text(encoding='utf-8'))
    print('--- JSON 샘플:', jsons[0].name)
    print(json.dumps(data, ensure_ascii=False, indent=1)[:3000])
else:
    import zipfile
    for z in sorted(raw.rglob('*.zip')):
        with zipfile.ZipFile(z) as zf:
            names = zf.namelist()
            print(f'{z.name}: {len(names)}개 항목, 예시 {names[:5]}')
            sample = next((n for n in names if n.lower().endswith('.json')), None)
            if sample:
                print('--- JSON 샘플:', sample)
                print(json.dumps(json.loads(zf.read(sample).decode('utf-8')), ensure_ascii=False, indent=1)[:3000])

In [ ]:
# 4. 학습 설정 — 필요하면 여기만 수정한다
# 모델은 YOLOv8n 고정: 휴대폰 온디바이스(LiteRT, INT8)에서 돌리는 것이 기준이라 더 큰 모델을 쓰지 않는다.
# 런타임은 G4(RTX PRO 6000, 96GB) > H100 > A100 > L4 > T4 순으로 빠르지만 아래 값은 어느 GPU 든 그대로 둔다.
MODEL = 'yolov8n.pt'
RAW_DIR = '/content/aihub_raw'   # 3번 셀에서 받은 라벨/원천 위치
CLASS_LIMIT = 100                # 학습할 클래스 수 (5번 셀에서 선정 방식 확정). 400종 전체는 혼동만 늘어 권장하지 않는다.
SAMPLES_PER_CLASS = 500          # 클래스당 이미지 수 상한 (0 = 전체)
VAL_RATIO = 0.15                 # 검증 데이터 비율
IMG_SIZE = 640                   # 앱 FoodDetector 전처리와 동일한 입력 크기
EPOCHS = 100                     # G4/A100 ≈ 1시간 안팎, L4 ≈ 2~3시간. patience 로 조기 종료된다.
PATIENCE = 20                    # 검증 mAP 가 20 에폭 동안 안 오르면 멈춘다
BATCH = 64                       # v8n@640 은 약 10GB 라 L4(24GB) 이상 어디서나 들어간다. T4(16GB) 에서 OOM 이면 32.
WORKERS = 8                      # 데이터 로더 스레드
CACHE = 'ram'                    # 640px 데이터셋 3~5GB 는 RAM 캐시가 가능하다(고용량 RAM 런타임). 메모리 부족이면 'disk'
SEED = 42

# Drive 에 결과를 보존한다 — 세션이 끊겨도 가중치·데이터셋이 남고 6번 셀의 RESUME 에 쓴다.
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS_DIR = '/content/drive/MyDrive/trex/runs'
else:
    RUNS_DIR = '/content/runs'

In [ ]:
# 5. YOLO 학습 형식으로 변환 — ⛔ 아직 작성 전
# 3번 셀 출력(라벨 JSON 의 클래스 필드·박스 좌표 형식·이미지 파일명 규칙)을 본 뒤에 완성한다.
# 완성되면 이 셀이: Validation 원천 묶음(대분류별 zip)을 하나씩 받아 → 목표 클래스만 640px 로 저장 + JSON 박스 → YOLO 라벨 → 원본 삭제
# 를 반복하고 data.yaml 을 만든다.
raise SystemExit('5번 변환 셀은 3번 출력을 보고 완성한다 — 지금은 실행하지 않는다')

In [ ]:
# 6. YOLOv8n 파인튜닝
# 세션이 끊겼으면 RESUME = True 로 바꿔 같은 셀을 다시 실행한다(가중치는 Drive 에 있다).
from pathlib import Path
from ultralytics import YOLO

RESUME = False
last = Path(RUNS_DIR) / 'food' / 'weights' / 'last.pt'
if RESUME and last.exists():
    model = YOLO(str(last))
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMG_SIZE,
        batch=BATCH,
        workers=WORKERS,
        cache=CACHE,
        cos_lr=True,
        seed=SEED,
        project=RUNS_DIR,
        name='food',
        exist_ok=True,
    )
BEST = str(Path(RUNS_DIR) / 'food' / 'weights' / 'best.pt')
print('best:', BEST)

In [ ]:
# 7. 검증 지표 확인 (mAP50이 0.8 이상이면 쓸 만하다). 헷갈리는 클래스는 RUNS_DIR/food/confusion_matrix.png 로 본다.
metrics = YOLO(BEST).val(data=str(data_yaml))
print('mAP50:', round(metrics.box.map50, 3), '/ mAP50-95:', round(metrics.box.map, 3))

In [ ]:
# 8. TFLite INT8 변환 — 입출력은 float32로 유지된다 (Ultralytics 기본 동작, 앱 요구사항)
# export 에 nms 인자를 주지 않는다(기본값). 앱 FoodDetector 는 raw 출력 (N, 4+클래스수, 8400) 을 기대한다.
from pathlib import Path
from ultralytics import YOLO

YOLO(BEST).export(format='tflite', int8=True, imgsz=IMG_SIZE, data=str(data_yaml))
tflite_candidates = sorted(Path(BEST).parent.rglob('*int8*.tflite'))
assert tflite_candidates, 'INT8 tflite 산출물을 찾지 못했다 — export 로그를 확인'
TFLITE_PATH = tflite_candidates[0]
print('변환 완료:', TFLITE_PATH)

In [ ]:
# 9. 앱 호환성 자동 검증 — FoodDetector가 기대하는 조건과 대조
import numpy as np
import tensorflow as tf

interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]

assert inp['dtype'] == np.float32, f"입력이 {inp['dtype']} — 앱은 float32 입력만 받는다 (FoodDetector 로드 시 거부됨)"
assert out['dtype'] == np.float32, f"출력이 {out['dtype']} — float32여야 한다"
# 앱은 NHWC [1,H,W,3] 과 NCHW [1,3,H,W] 를 모두 처리한다. 어느 쪽인지 여기서 확인해 둔다.
shape = list(inp['shape'])
if shape == [1, IMG_SIZE, IMG_SIZE, 3]:
    layout = 'NHWC'
elif shape == [1, 3, IMG_SIZE, IMG_SIZE]:
    layout = 'NCHW'
else:
    raise AssertionError(f"입력 형태 {shape} — [1,{IMG_SIZE},{IMG_SIZE},3] 또는 [1,3,{IMG_SIZE},{IMG_SIZE}] 이어야 한다")
nc = len(class_names)
assert nc + 4 in list(out['shape']), f"출력 형태 {out['shape']}가 클래스 수 {nc}(+4)와 맞지 않는다"
print(f"검증 통과 — 입력 {shape} float32 ({layout}), 출력 {out['shape']} (클래스 {nc}종)")

In [ ]:
# 10. 앱 배포용 파일 패키징 + 다운로드
import shutil
from google.colab import files

out_dir = Path('/content/app_assets')
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir()
shutil.copy(TFLITE_PATH, out_dir / 'yolov8n_food.tflite')
# 클래스 인덱스 순서 그대로 — 앱 food_labels.txt 형식 (전부 음식이므로 '#' 접두어 없음)
(out_dir / 'food_labels.txt').write_text('\n'.join(class_names) + '\n', encoding='utf-8')

shutil.make_archive('/content/trex_food_model', 'zip', out_dir)
files.download('/content/trex_food_model.zip')
print('완료 — zip 안의 두 파일을 TREX_UI/app/src/main/assets/models/ 에 덮어쓰고 빌드한다')

### 적용 방법
1. 받은 `trex_food_model.zip`을 풀어 `yolov8n_food.tflite`, `food_labels.txt` 두 파일을 `TREX_UI/app/src/main/assets/models/`에 **덮어쓴다**.
2. `TrexData.kt` 의 `foodDatabase` 에 새 라벨의 영양값을 추가한다(라벨명 = DB 키). 영양DB.xlsx 가 있으니 근사값 대신 실제 값을 넣는다.
3. 앱을 빌드·설치하면 FoodDetector가 모델을 발견하고 실추론으로 동작한다. 로드 시 로그에 입력·출력 형태가 찍힌다.

### 런타임 안내 (Colab Pro+, 2026-09 기준)
- 런타임 유형 변경에서 고를 수 있는 GPU: **G4(NVIDIA RTX PRO 6000 Blackwell, 96GB — 가장 빠름)**, H100, A100(고용량 RAM 켜면 80GB), L4, T4. G4·H100 은 배정이 안 될 때가 있으니 **G4 → A100 → L4** 순으로 잡히는 것을 쓴다. 1번 셀 출력으로 무엇을 받았는지 확인한다.
- YOLOv8n 학습은 GPU 를 크게 타지 않는다 — G4 라고 배치를 키울 필요 없고 에폭당 시간만 줄어든다.
- Pro+ 는 최대 24시간 세션·백그라운드 실행을 지원하지만 컴퓨팅 단위가 소진되면 끊긴다. 결과는 Drive(`RUNS_DIR`)에 남으므로 6번 셀 `RESUME = True` 로 이어서 학습한다.

### 문제 해결
- **OOM**: 4번 셀 `BATCH = 32`(T4). 그래도 나면 `CACHE = 'disk'`.
- **GPU 세션 끊김**: 6번 셀 `RESUME = True` 후 재실행(가중치는 Drive 에 있다).
- **정확도 낮음**: `SAMPLES_PER_CLASS`·`EPOCHS`를 늘리거나 `CLASS_LIMIT`을 줄여 클래스를 압축한다. 헷갈리는 클래스(예: 콩나물국↔계란국)는 7번 셀의 confusion matrix 로 확인.
- **9번 셀 assert 실패(입출력이 float32가 아님)**: 8번 셀에서 `int8=True`를 `int8=False`로 바꿔 float32로 다시 변환한다(파일 2~3배). 산출물 검색 패턴도 `*int8*.tflite` 대신 `*.tflite`로.
- **9번 셀에서 출력이 `[1, 300, 6]`**: NMS 가 포함된 형태로 export 된 것. 8번 셀 export 에 `nms=` 인자가 들어가 있지 않은지 확인하고 빼서 다시 변환한다.